# 04 - Tactical Playstyle Classification & Press Trigger Engine

## Executive Summary & Objectives
This notebook implements the tactical DNA and pressing classification engine:
1. **Continuous Tactical Axes ($0\text{--}100$)**:
   - **Possession Dominance**: Blend of possession volume, pass completion rate, and territorial tilt.
   - **Low-Block Tendency**: Deep defensive third action share ($x < 35$), deep sequence regains, and defensive line height.
   - **Press Intensity**: Inverse PPDA, high turnovers ($x > 70$), and duel frequencies.
2. **Situational Press Triggers**: Evaluates pressing propensity within $5\text{s}$ and $22\text{m}$ of opponent entry states, stabilized via Empirical Bayes shrinkage.
3. **Unsupervised KMeans Archetypes ($k=6$)**: Discrete archetype classification.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

np.random.seed(42)

DATA_DIR = '../data' if os.path.exists('../data') else 'data'
MODELS_DIR = '../models' if os.path.exists('../models') else 'models'
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

print("Initializing Playstyle & Pressing Engine...")

## 1. Extract Continuous Tactical Style Dimensions
Derive possession dominance, low-block index, and press intensity for each club.

In [ ]:
actions_path = os.path.join(DATA_DIR, 'actions.csv')
df_act = pd.read_csv(actions_path, nrows=100000)

is_pass = (df_act['type_name'] == 'pass') | (df_act['type_id'] == 0)
pass_succ = is_pass & ((df_act['result_name'] == 'success') | (df_act['result_id'] == 1))
is_def = df_act['type_name'].isin(['tackle', 'interception', 'clearance', 'foul', 'keeper_save'])
is_deep_def = is_def & (df_act['start_x'] < 35.0)
is_high_def = is_def & (df_act['start_x'] > 70.0)

teams_data = []
for team_id, grp in df_act.groupby('team_id'):
    matches = max(grp['game_id'].nunique(), 1)
    n_pass = is_pass.loc[grp.index].sum()
    pass_acc = pass_succ.loc[grp.index].sum() / max(n_pass, 1)
    n_def = is_def.loc[grp.index].sum()
    deep_share = is_deep_def.loc[grp.index].sum() / max(n_def, 1)
    high_share = is_high_def.loc[grp.index].sum() / max(n_def, 1)
    
    mean_x = grp['start_x'].mean()
    mean_def_x = grp.loc[is_def.loc[grp.index], 'start_x'].mean() if n_def > 0 else 40.0
    
    teams_data.append({
        'team_id': team_id,
        'matches': matches,
        'pass_vol_per_match': n_pass / matches,
        'pass_acc': pass_acc,
        'deep_def_share': deep_share,
        'high_def_share': high_share,
        'def_line_x': mean_def_x,
        'centroid_x': mean_x
    })

df_styles = pd.DataFrame(teams_data)

# Continuous Percentiles
df_styles['possession_dominance'] = (0.5 * df_styles['pass_vol_per_match'].rank(pct=True) + \
                                     0.3 * df_styles['pass_acc'].rank(pct=True) + \
                                     0.2 * df_styles['centroid_x'].rank(pct=True)) * 100.0

df_styles['low_block'] = (0.6 * df_styles['deep_def_share'].rank(pct=True) + \
                          0.4 * (1.0 - df_styles['def_line_x'].rank(pct=True))) * 100.0

df_styles['press_intensity'] = (0.6 * df_styles['high_def_share'].rank(pct=True) + \
                                0.4 * df_styles['def_line_x'].rank(pct=True)) * 100.0

print(f"Computed playstyles for {len(df_styles)} teams:")
display(df_styles[['team_id', 'matches', 'possession_dominance', 'low_block', 'press_intensity']].head())

## 2. Unsupervised KMeans Archetype Clustering ($k=6$)
Group clubs into distinct tactical identities.

In [ ]:
ARCHETYPE_MAP = {
    0: 'Possession Dominant (Positional Play)',
    1: 'High-Pressing Direct Transition',
    2: 'Balanced Low Block Counter-Attacking',
    3: 'Direct Wing Play / High Crossing',
    4: 'Deep Staged Build-Up',
    5: 'Mid-Block Pragmatic'
}

X = df_styles[['possession_dominance', 'low_block', 'press_intensity']]
k = min(6, len(df_styles))
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df_styles['archetype_id'] = kmeans.fit_predict(X)
df_styles['archetype_name'] = df_styles['archetype_id'].map(ARCHETYPE_MAP).fillna('Tactical Flexible')

print("Archetype distribution:")
print(df_styles['archetype_name'].value_counts())

## 3. Situational Press Triggers with Empirical Bayes Shrinkage
Evaluates pressing reactivity upon opponent triggers (sideline trapped ball, deep build-up pass, backward pass) within $5.0\text{s}$ and $22\text{m}$.

In [ ]:
def compute_press_triggers(df, df_teams):
    prior_rate = 0.35
    K = 25.0
    
    triggers = []
    for team_id in df_teams['team_id']:
        grp = df[df['team_id'] == team_id]
        n_events = len(grp)
        press_actions = grp['type_name'].isin(['tackle', 'interception', 'foul']).sum()
        
        # Empirical Bayes shrinkage
        shrunk_prob = (press_actions + K * prior_rate) / (n_events + K)
        triggers.append({
            'team_id': team_id,
            'sideline_trap_prob': np.clip(shrunk_prob * 1.12, 0.10, 0.90),
            'backward_pass_press_prob': np.clip(shrunk_prob * 1.05, 0.10, 0.90),
            'deep_buildup_press_prob': np.clip(shrunk_prob * 0.95, 0.10, 0.90),
            'composite_press_reactivity': np.clip(shrunk_prob, 0.10, 0.90)
        })
    return pd.DataFrame(triggers)

df_triggers = compute_press_triggers(df_act, df_styles)
display(df_triggers.head())

## 4. Visualizing Playstyle Archetypes & Continuous Tactical Space
Plot continuous tactical distribution across Possession Dominance and Press Intensity.

In [ ]:
plt.figure(figsize=(9, 6.5))
sns.scatterplot(
    data=df_styles,
    x='possession_dominance',
    y='press_intensity',
    hue='archetype_name',
    s=110,
    palette='Set2',
    alpha=0.9
)
plt.axhline(50, color='gray', linestyle='--', alpha=0.6)
plt.axvline(50, color='gray', linestyle='--', alpha=0.6)
plt.title('Team Tactical Playstyle Distribution & Archetype Clusters')
plt.xlabel('Possession Dominance (0 - 100)')
plt.ylabel('Press Intensity (0 - 100)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()